# 07 — System Integration
**NewsBot Intelligence System 2.0 | ITAI 2373**

End-to-end pipeline: load saved models, classify new articles, run full NLP + LLM analysis, and generate system report.

In [18]:
!pip install joblib scikit-learn vaderSentiment spacy groq -q

In [19]:
from google.colab import drive
drive.mount('/content/data')
PROJECT = '/content/data/MyDrive/ITAI2373-NewsBot-Final'

Drive already mounted at /content/data; to attempt to forcibly remount, call drive.mount("/content/data", force_remount=True).


In [20]:
# ── Verify df_final ───────────────────────────────────────────────
import os, pandas as pd
if os.path.exists(f'{PROJECT}/data/processed/df_final.pkl'):
    df_final = pd.read_pickle(f'{PROJECT}/data/processed/df_final.pkl')
    print(f'Loaded df_final: {df_final.shape}')
    print(f'Columns: {list(df_final.columns)}')
else:
    raise FileNotFoundError(
        'df_final.pkl not found.\n'
        'Run notebooks 01 and 02 first and make sure it saved df_final.'
    )

Loaded df_final: (1490, 21)
Columns: ['ArticleId', 'Text', 'Category', 'text_length', 'word_count', 'doc_id', 'cleaned_text', 'entities', 'sentiment_compound', 'sentiment_label', 'passive_voice_instance_count', 'top_tfidf', 'lda_dominant_topic', 'lda_topic_confidence', 'lda_topic_label', 'nmf_dominant_topic', 'nmf_topic_confidence', 'nmf_topic_label', 'cluster', 'detected_language', 'lang_confidence']


In [21]:
import joblib, json, os, requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from google.colab import userdata

plt.rcParams.update({
    'figure.facecolor':'#0f1117','axes.facecolor':'#1a1d2e',
    'axes.edgecolor':'#2d3561','axes.labelcolor':'#e8d5a3',
    'xtick.color':'#a0a8c0','ytick.color':'#a0a8c0',
    'text.color':'#e8d5a3','grid.color':'#2d3561','grid.alpha':0.5,'figure.dpi':120
})
PALETTE = {'tech':'#4fc3f7','business':'#81c784','politics':'#e57373',
           'sport':'#ffb74d','entertainment':'#ce93d8'}
# OLLAMA_HOST  = 'http://127.0.0.1:11434'
# OLLAMA_MODEL = 'llama3.2:1b'
vader        = SentimentIntensityAnalyzer()

GROQ_API_KEY = userdata.get('GROQ_API_KEY') # Securely fetch API key
GROQ_MODEL_NAME = 'llama-3.1-8b-instant'

print('System integration setup complete.')


System integration setup complete.


## 7.1 — Load Saved Models

In [22]:
# Load trained classifier and vectorizer from data/models/
MODELS_DIR = f'{PROJECT}/data/models/'

clf     = joblib.load(f'{MODELS_DIR}/classifier.pkl')
tfidf   = joblib.load(f'{MODELS_DIR}/tfidf_vectorizer.pkl')

with open(f'{MODELS_DIR}/model_info.json') as f:
    model_info = json.load(f)

print('Loaded models:')
print(f'  Classifier  : {model_info["model_type"]}')
print(f'  Accuracy    : {model_info["accuracy"]}')
print(f'  CV Score    : {model_info["cv_mean"]} +/- {model_info["cv_std"]}')
print(f'  Trained on  : {model_info["n_train"]} articles')
print(f'  Vocab size  : {model_info["vocab_size"]:,} terms')


Loaded models:
  Classifier  : LogisticRegression
  Accuracy    : 0.9698
  CV Score    : 0.9758 +/- 0.0081
  Trained on  : 1192 articles
  Vocab size  : 5,000 terms


## 7.2 — Classify New Articles

In [23]:
def classify_article(text):
    """Classify a new article using the saved trained model."""
    import re
    cleaned = text.lower()
    cleaned = re.sub(r'http\S+', ' ', cleaned)
    cleaned = re.sub(r'[^a-z\s]', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    X    = tfidf.transform([cleaned])
    pred = clf.predict(X)[0]
    prob = clf.predict_proba(X)[0]
    conf = prob.max()

    scores = vader.polarity_scores(cleaned)
    c = scores['compound']
    sentiment = 'Positive' if c>=0.05 else 'Negative' if c<=-0.05 else 'Neutral'

    return {
        'predicted_category': pred,
        'confidence':         round(float(conf), 4),
        'all_probabilities':  dict(zip(clf.classes_, [round(float(p),4) for p in prob])),
        'sentiment_label':    sentiment,
        'sentiment_compound': round(c, 4),
        'word_count':         len(cleaned.split()),
    }

# Test on unseen articles
test_articles = [
    ('Should be TECH',
     'Google DeepMind researchers published a breakthrough in protein structure prediction using transformer models, potentially revolutionizing drug discovery and materials science.'),
    ('Should be SPORT',
     'Serena Williams announced her retirement from professional tennis after a decorated career spanning four decades and 23 Grand Slam singles titles.'),
    ('Should be BUSINESS',
     'The Federal Reserve signaled three interest rate cuts in 2025 as inflation fell toward its 2% target, boosting stock markets globally and relieving pressure on mortgage holders.'),
    ('Should be POLITICS',
     'Parliament passed the new digital rights legislation by a narrow majority after heated debate over surveillance provisions and data privacy protections.'),
    ('Should be ENTERTAINMENT',
     'The Cannes Film Festival announced its competition lineup featuring directors from twelve countries, with a focus on climate change and social justice themes.'),
]

print('7.2 — Classification of Unseen Articles')
print('='*60)
for expected, text in test_articles:
    result = classify_article(text)
    status = '✓' if expected.split()[-1].lower() == result['predicted_category'] else '✗'
    print(f'{status} Expected: {expected}')
    print(f'  Predicted  : {result["predicted_category"]} ({result["confidence"]*100:.1f}% confidence)')
    print(f'  Sentiment  : {result["sentiment_label"]} ({result["sentiment_compound"]:+.3f})')
    print()


7.2 — Classification of Unseen Articles
✓ Expected: Should be TECH
  Predicted  : tech (27.5% confidence)
  Sentiment  : Neutral (+0.000)

✓ Expected: Should be SPORT
  Predicted  : sport (46.1% confidence)
  Sentiment  : Positive (+0.103)

✓ Expected: Should be BUSINESS
  Predicted  : business (52.1% confidence)
  Sentiment  : Positive (+0.542)

✗ Expected: Should be POLITICS
  Predicted  : tech (28.4% confidence)
  Sentiment  : Neutral (+0.000)

✓ Expected: Should be ENTERTAINMENT
  Predicted  : entertainment (42.3% confidence)
  Sentiment  : Positive (+0.765)



## 7.3 — Full Pipeline on New Article

In [24]:
def call_llm(prompt, system=None):
    from groq import Groq
    messages = []
    if system: messages.append({'role':'system','content':system})
    messages.append({'role':'user','content':prompt})
    try:
        client = Groq(api_key=GROQ_API_KEY)
        chat_completion = client.chat.completions.create(
            messages=messages,
            model=GROQ_MODEL_NAME,
            temperature=0.3,
        )
        return chat_completion.choices[0].message.content.strip()
    except Exception as e:
        return f'[LLM unavailable: {e}]'

def full_pipeline(article_text):
    """Run complete NewsBot 2.0 pipeline on a single article."""
    import re, spacy
    nlp = spacy.load('en_core_web_sm')

    # 1. Classify
    clf_result = classify_article(article_text)

    # 2. NER
    doc = nlp(article_text[:5000])
    entities = {}
    for ent in doc.ents:
        entities.setdefault(ent.label_,[]).append(ent.text.strip())

    # 3. Summarize via LLM
    summary_prompt = (f'Summarize this {clf_result["predicted_category"]} article in 3 sentences. \
                      Output only the summary.\n\n{article_text[:2500]}')
    summary = call_llm(summary_prompt)

    # 4. Key insights
    insights_prompt = (f'List 3 key findings from this article as a JSON array of strings. \
                       Output ONLY the JSON array.\n\n{article_text[:2000]}')
    raw = call_llm(insights_prompt)
    try:
        insights = json.loads(raw.strip().lstrip('```json').rstrip('```'))
    except:
        insights = [raw]

    return {
        'classification':  clf_result,
        'entities':        entities,
        'summary':         summary,
        'key_insights':    insights,
    }

# Run full pipeline
print('7.3 — Full Pipeline Demo')
print('='*60)
new_article = """
OpenAI and Microsoft announced a major expansion of their partnership,
with the software giant committing an additional $10 billion in investment
to accelerate development of artificial general intelligence systems.
The deal gives Microsoft exclusive rights to commercialize OpenAI's technologies
through its Azure cloud platform. Critics raised concerns about market concentration
in the AI sector, while investors pushed both companies' valuations to record highs.
The partnership has produced tools including GitHub Copilot and Azure OpenAI Service,
which are now used by more million enterprise customers worldwide.
"""

result = full_pipeline(new_article.strip())
print(f'Category   : {result["classification"]["predicted_category"]} ({result["classification"]["confidence"]*100:.1f}%)')
print(f'Sentiment  : {result["classification"]["sentiment_label"]} ({result["classification"]["sentiment_compound"]:.3f})')
print(f'\nKey Entities:')
for etype, ents in list(result['entities'].items())[:4]:
    print(f'  {etype}: {list(set(ents))[:3]}')
print(f'\nSummary:')
print(f'  {result["summary"]}')
print(f'\nKey Insights:')
for ins in result['key_insights']:
    print(f'  - {ins}')


7.3 — Full Pipeline Demo
Category   : business (37.0%)
Sentiment  : Positive (0.402)

Key Entities:
  ORG: ['GitHub Copilot', 'Microsoft', 'AI']
  MONEY: ['an additional $10 billion']
  PERSON: ['OpenAI']
  CARDINAL: ['more million']

Summary:
  Microsoft has invested an additional $10 billion in OpenAI to accelerate the development of artificial general intelligence systems. The deal grants Microsoft exclusive rights to commercialize OpenAI's technologies through its Azure cloud platform. The partnership has already produced successful tools, including GitHub Copilot and Azure OpenAI Service, used by millions of enterprise customers worldwide.

Key Insights:
  - Microsoft committed an additional $10 billion in investment to accelerate development of artificial general intelligence systems.
  - Microsoft gained exclusive rights to commercialize OpenAI's technologies through its Azure cloud platform.
  - The partnership has produced tools including GitHub Copilot and Azure OpenAI Servic

## 7.4 — System Report

In [25]:
try:
    _ = df_final
    has_df = True
except NameError:
    has_df = False

print('='*60)
print('  NEWSBOT INTELLIGENCE SYSTEM 2.0 — FINAL REPORT')
print('='*60)
if has_df:
    print(f'  Dataset         : BBC News ({len(df_final):,} articles)')
    print(f'  Categories      : {sorted(df_final.Category.unique().tolist())}')
    if 'sentiment_label' in df_final.columns:
        dist = df_final.sentiment_label.value_counts().to_dict()
        print(f'  Sentiment dist  : {dist}')
    if 'detected_language' in df_final.columns:
        en_pct = (df_final.detected_language=='en').mean()*100
        print(f'  English articles: {en_pct:.1f}%')

print(f'\n  Model Performance:')
print(f'  Classifier      : {model_info["model_type"]}')
print(f'  Test Accuracy   : {model_info["accuracy"]*100:.2f}%')
print(f'  CV Accuracy     : {model_info["cv_mean"]*100:.2f}% +/- {model_info["cv_std"]*100:.2f}%')
print(f'  Vocab size      : {model_info["vocab_size"]:,}')
print(f'\n  System Modules:')
print(f'  Module A: Topic Modeling    (LDA + NMF, 10 topics)')
print(f'  Module B: Language Models   (Groq/{GROQ_MODEL_NAME})')
print(f'  Module C: Multilingual      (langdetect + deep-translator)')
print(f'  Module D: Conversational    (rule-based intent + LLM)')
print(f'  Bonus   : Web App           (Flask + HTML/JS frontend)')
print('='*60)

  NEWSBOT INTELLIGENCE SYSTEM 2.0 — FINAL REPORT
  Dataset         : BBC News (1,490 articles)
  Categories      : ['business', 'entertainment', 'politics', 'sport', 'tech']
  Sentiment dist  : {'Positive': 1054, 'Negative': 426, 'Neutral': 10}
  English articles: 100.0%

  Model Performance:
  Classifier      : LogisticRegression
  Test Accuracy   : 96.98%
  CV Accuracy     : 97.58% +/- 0.81%
  Vocab size      : 5,000

  System Modules:
  Module A: Topic Modeling    (LDA + NMF, 10 topics)
  Module B: Language Models   (Groq/llama-3.1-8b-instant)
  Module C: Multilingual      (langdetect + deep-translator)
  Module D: Conversational    (rule-based intent + LLM)
  Bonus   : Web App           (Flask + HTML/JS frontend)


In [26]:
# Save system report
MODEL_DIR = f'{PROJECT}/data/results/'
report = {
    'model_info':   model_info,
    'modules':      ['A - Topic Modeling','B - Language Models',
                     'C - Multilingual','D - Conversational Interface'],
    'web_app':      'Flask (app.py) + HTML standalone',
    'llm_model':    GROQ_MODEL_NAME,
}
os.makedirs(MODEL_DIR, exist_ok=True)
with open(f'{MODEL_DIR}/system_report.json','w') as f:
    json.dump(report, f, indent=2)
print('System report saved to data/results/system_report.json')


System report saved to data/results/system_report.json


In [27]:
# ── Save df_final for next notebook ──────────────────────────────
df_final.to_pickle(f'{PROJECT}/data/processed/df_final.pkl')
print(f'Saved df_final: {df_final.shape} → {PROJECT}/data/processed/df_final.pkl')
print(f'Columns: {list(df_final.columns)}')

Saved df_final: (1490, 21) → /content/data/MyDrive/ITAI2373-NewsBot-Final/data/processed/df_final.pkl
Columns: ['ArticleId', 'Text', 'Category', 'text_length', 'word_count', 'doc_id', 'cleaned_text', 'entities', 'sentiment_compound', 'sentiment_label', 'passive_voice_instance_count', 'top_tfidf', 'lda_dominant_topic', 'lda_topic_confidence', 'lda_topic_label', 'nmf_dominant_topic', 'nmf_topic_confidence', 'nmf_topic_label', 'cluster', 'detected_language', 'lang_confidence']


### Download to use on local LLM

In [29]:
import shutil
shutil.make_archive('/content/data/MyDrive/ITAI2373-NewsBot-Final/newsbot_models',
                    'zip',
                    '/content/data/MyDrive/ITAI2373-NewsBot-Final/data')
print("Zipped! Download from Google Drive.")

Zipped! Download from Google Drive.
